# 02 — Compute cache (Slice 1b, `cultureQC_upgrade.md` §4A)

**The big GPU pass.** Runs Cellpose-SAM, the QC EfficientNet-B0, and DINOv2 once
over every image this project needs and stores **raw outputs** (prob maps, logits,
embeddings, quality metrics) keyed by `(image_sha256, crop_spec, model_name,
model_version)`. Everything downstream — thresholds, calibration, binning, crop
choices — reads from this cache on CPU, with zero recompute. Rule 10: no second
GPU pass after this one except explicitly-called-out cases, so this notebook
**stops for your approval** after a 100-image budget test before running the
full pass.

Pins `slice-1b-compute-cache` at `53605a0`. Run **`nb/00_download_datasets.ipynb`
first** — this notebook assumes everything it fetches is already on Drive.

**Image sets in this pass** (see `docs/DATASETS.md` for full provenance):
| Set | n images | Role |
|---|---:|---|
| `data/tiles/` (synthetic) | 4,000 | anomaly banks, calibration, per-class AUROC |
| EVICAN `eval2019` | 98 | held-out real-data eval, bank calibration on real normals |
| AutoQC-Bench `test/` | 148 | external anomaly benchmark eval |

**Not in this pass** (see `docs/DATASETS.md`): C2C12 (not locally available — too
large, no tested fetcher yet), adherent Cell Tracking Challenge (blocked on
permission), shifted synthetic tiles (§8.2 — the EVICAN+C2C12-background
generator doesn't exist yet; that's Slice 4b work). Adding these later is cache
*growth*, not a second pass over already-cached images — `Cache.build()` skips
any `(image_sha256, ...)` key already present, so running `nb/03_cache_additions.ipynb`
against this same cache once those are ready costs nothing for what's done here.

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/cultureqc'

In [ ]:
REPO_URL = 'https://github.com/n1tishc/cultureqc.git'
BRANCH = 'slice-1b-compute-cache'
PINNED_SHA = '53605a0'

!rm -rf /content/cultureqc
!git clone --branch $BRANCH $REPO_URL /content/cultureqc
%cd /content/cultureqc
!git checkout $PINNED_SHA
!git rev-parse HEAD

In [ ]:
!pip install -q -r requirements.txt

import torch
print('torch', torch.__version__, '— CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU — Runtime > Change runtime type > GPU'

In [ ]:
import os, sys, time
sys.path.insert(0, '/content/cultureqc')
import pandas as pd

from culture.cache import Cache, ImageRecord, image_sha256

EVICAN_DIR = f'{DRIVE_ROOT}/data/sources/evican/eval2019_images'
TILES_DIR = f'{DRIVE_ROOT}/data/tiles'
AUTOQC_DIR = f'{DRIVE_ROOT}/data/sources/autoqc_bench'
CACHE_DIR = f'{DRIVE_ROOT}/cache'

for p in [EVICAN_DIR, TILES_DIR, AUTOQC_DIR]:
    assert os.path.isdir(p), f'{p} missing — run nb/00_download_datasets.ipynb first'
print('all input datasets present')

## Build the image record list

One `ImageRecord` per image, tagged by dataset. `keep_full_patches_for` follows
§4A.3's storage rule: full DINOv2 patch embeddings only for bank-source normals
and eval sets (EVICAN, AutoQC-Bench `test/`, and the synthetic tiles' `normal`
class) — everything else gets CLS-only.

In [ ]:
tiles_manifest = pd.read_csv(f'{TILES_DIR}/manifest.csv')
print(tiles_manifest.columns.tolist())
print(tiles_manifest.head(2))

⚠️ **Check the printed columns/row above** before continuing — the next cell
assumes `manifest.csv`'s `tile_id` already includes its extension (e.g.
`normal_00000.png`) and that tiles live at `{TILES_DIR}/{class}/{tile_id}`
(class-subdirectory layout, matching `scripts/synth_contamination.py`'s actual
output as of the pinned commit — verified against the Mac's local `data/tiles/`:
`contamination_suspected/`, `detachment/`, `image_quality/`, `normal/`). If the
printed layout differs, fix the path-building line below before running the full
pass — this is exactly the kind of assumption worth catching on the 100-image
budget test, not after a multi-hour full run.

In [ ]:
records = []
keep_full_patches_for = set()

# -- synthetic tiles (data/tiles/<class>/<tile_id>, tile_id includes extension) --
for _, row in tiles_manifest.iterrows():
    path = os.path.join(TILES_DIR, row['class'], row['tile_id'])
    if not os.path.exists(path):
        continue
    rec = ImageRecord(path=path, dataset='synth_tiles', source_path=path)
    records.append(rec)
    if row['class'] == 'normal':
        keep_full_patches_for.add(image_sha256(path))

print(len(records), 'synthetic tile records so far')

In [ ]:
# -- EVICAN eval2019 (held-out real-data eval — full patches) --
n_before = len(records)
for fn in sorted(os.listdir(EVICAN_DIR)):
    if not fn.lower().endswith('.jpg'):
        continue
    path = os.path.join(EVICAN_DIR, fn)
    records.append(ImageRecord(path=path, dataset='evican_eval2019', source_path=path))
    keep_full_patches_for.add(image_sha256(path))
print(len(records) - n_before, 'EVICAN records')

In [ ]:
# -- AutoQC-Bench test/ (external anomaly eval — full patches) --
# test/anomalies/ is nested by anomaly type then species
# (artifacts|illumination|z-shift|contamination|air_bubble)/(human|mouse)/,
# not flat — walk recursively rather than assuming a fixed depth.
n_before = len(records)
test_root = os.path.join(AUTOQC_DIR, 'test')
for root, _dirs, files in os.walk(test_root):
    for fn in sorted(files):
        if not fn.lower().endswith(('.tif', '.tiff')):
            continue
        path = os.path.join(root, fn)
        records.append(ImageRecord(path=path, dataset='autoqc_bench_test', source_path=path))
        keep_full_patches_for.add(image_sha256(path))
print(len(records) - n_before, 'AutoQC-Bench test records')
print()
print(len(records), 'total images |', len(keep_full_patches_for), 'get full patch embeddings')

## Budget test — 100 images (§4A.3: run first, report timing, ⏸ wait for approval)

A stratified sample (not just the first 100, which would be all-synthetic-tiles)
so the projection reflects all three image sets' actual per-model cost.

In [ ]:
import random
random.seed(0)

by_dataset = {}
for r in records:
    by_dataset.setdefault(r.dataset, []).append(r)

N_BUDGET = 100
frac = {ds: len(rs) / len(records) for ds, rs in by_dataset.items()}
budget_records = []
for ds, rs in by_dataset.items():
    n = max(1, round(frac[ds] * N_BUDGET))
    budget_records += random.sample(rs, min(n, len(rs)))
budget_records = budget_records[:N_BUDGET]
print(len(budget_records), 'budget-test images:',
      {ds: sum(1 for r in budget_records if r.dataset == ds) for ds in by_dataset})

In [ ]:
BUDGET_CACHE_DIR = '/content/cache_budget_test'
!rm -rf $BUDGET_CACHE_DIR
budget_cache = Cache(BUDGET_CACHE_DIR)

t0 = time.time()
summary = budget_cache.build(
    budget_records,
    models=('seg', 'qc', 'quality', 'dino'),
    crop_fracs=(0.25, 0.5),
    crops_per_frac=8,
    keep_full_patches_for={image_sha256(r.path) for r in budget_records} & keep_full_patches_for,
)
wall_s = time.time() - t0

print(f'{wall_s:.1f}s wall for {summary["n_images"]} images')
print('per-image seconds by model:', summary['per_image_s'])

In [ ]:
import subprocess

du_out = subprocess.run(['du', '-sb', BUDGET_CACHE_DIR], capture_output=True, text=True).stdout
budget_bytes = int(du_out.split()[0])
n_full = len(budget_records)
n_total = len(records)

projected_wall_s = wall_s * (n_total / n_full)
projected_bytes = budget_bytes * (n_total / n_full)

print(f'Budget test: {n_full} images, {wall_s:.1f}s, {budget_bytes/1e6:.1f} MB')
print(f'Full pass ({n_total} images) projection:')
print(f'  runtime: ~{projected_wall_s/60:.1f} min ({projected_wall_s/3600:.2f} hr)')
print(f'  storage: ~{projected_bytes/1e9:.2f} GB')
print()
print('Per-model per-image cost (seconds):')
for model, s in summary['per_image_s'].items():
    print(f'  {model:10s} {s:.3f}s/image  ->  ~{s * n_total / 60:.1f} min for the full set')

## ⏸ STOP — review the numbers above before continuing

Per §4A.3: *"Budget first ... Wait for approval before the full run."* Read the
projected runtime and storage printed above. If they look right for your Colab
session/disk budget, flip the flag in the next cell to `True` and run the rest of
the notebook. **Do not just "run all"** through this point — that defeats the
point of the budget test.

In [ ]:
RUN_FULL_PASS = False  # <-- flip to True only after reading the budget-test numbers above

if not RUN_FULL_PASS:
    raise SystemExit(
        'Stopped at the Slice 1b budget-test checkpoint (spec §4A.3). '
        'Review the projected runtime/storage in the cell above, then set '
        'RUN_FULL_PASS = True and re-run this cell to continue.'
    )
print('Proceeding to the full pass.')

## Full pass

Writes directly to the Drive-backed `CACHE_DIR` — resumable by construction
(`Cache.build()` skips any `(image_sha256, crop_spec, model_name, model_version)`
already on disk), so a Colab disconnect just means re-running this cell.

In [ ]:
cache = Cache(CACHE_DIR)

t0 = time.time()
full_summary = cache.build(
    records,
    models=('seg', 'qc', 'quality', 'dino'),
    crop_fracs=(0.25, 0.5),
    crops_per_frac=8,
    keep_full_patches_for=keep_full_patches_for,
)
wall_s = time.time() - t0
print(f'Full pass: {full_summary["n_images"]} images in {wall_s/60:.1f} min')
print('timings_s:', full_summary['timings_s'])

If this cell is interrupted (disconnect, timeout) — just re-run it. Already-
cached images are skipped (that's what the resume/idempotency test in
`tests/test_cache_parity.py` verifies on the Mac before this notebook ever runs).

## Manifest + slim export

In [ ]:
import json
manifest = json.load(open(os.path.join(CACHE_DIR, 'MANIFEST.json')))
print(json.dumps(manifest, indent=2))

In [ ]:
SLIM_DIR = f'{DRIVE_ROOT}/cache_slim'
cache.export_slim(SLIM_DIR)

full_bytes = int(subprocess.run(['du', '-sb', CACHE_DIR], capture_output=True, text=True).stdout.split()[0])
slim_bytes = int(subprocess.run(['du', '-sb', SLIM_DIR], capture_output=True, text=True).stdout.split()[0])
print(f'full cache: {full_bytes/1e9:.2f} GB   slim export: {slim_bytes/1e9:.2f} GB')

## ⏸ Report back (per §4A.4)

Copy the printed runtime, storage, row counts (manifest above), and this cell's
output into the Slice 1b write-up. Acceptance per §4A.4:
- [x] Full cache built (this notebook)
- [x] Manifest complete (printed above)
- [ ] Parity test green — already verified on the Mac (`tests/test_cache_parity.py`,
      4/4 pass) *before* this pass, per rule 10
- [ ] FOV-noise analysis from cached crops — Mac-side follow-up, `scripts/`, no GPU
      needed (reads `cache.load_confluency(crop_spec=...)` from the slim export)
- [ ] Slim export `load_*` works on the Mac without a GPU — verify after downloading
      `cache_slim/` from Drive

**Next step (on the Mac, not here):** download `DRIVE_ROOT/cache_slim/` into the
repo at `cache/` (e.g. Google Drive Desktop sync, or `rclone copy`), then run
`cache.load_confluency()` / `load_logits()` / `load_embeddings()` locally to
confirm the slim export is self-contained and GPU-free.

In [ ]:
print('n_images (full pass):', full_summary['n_images'])
print('row_counts:', manifest['row_counts'])
print('model_versions:', manifest['model_versions'])
print('full cache size: {:.2f} GB'.format(full_bytes / 1e9))
print('slim export size: {:.2f} GB'.format(slim_bytes / 1e9))
print('wall time: {:.1f} min'.format(wall_s / 60))